# Шаг 1: добавление контрольных переменных World Bank

**Цель:** Скачать контрольные переменные из World Bank WDI и добавить их в панель экспорта.

**Метод:** JSON/CSV API World Bank — публичный, не требует ключа. Три показателя: ВВП (NY.GDP.MKTP.CD), добавленная стоимость обрабатывающей промышленности (NV.IND.MANF.CD), её доля в ВВП (NV.IND.MANF.ZS).

**Входные данные:** `../data/clean_china_ree_exports_284690_2010_2024.xlsx`  
**Выходные данные:** `../data/china_ree_exports_with_controls.xlsx`  

**Запускать первым** перед любыми регрессионными ноутбуками.

## 0. Импорты и конфигурация

In [1]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import statsmodels.formula.api as smf

INPUT_FILE  = "../data/clean_china_ree_exports_284690_2010_2024.xlsx"
OUTPUT_FILE = "../data/china_ree_exports_with_controls.xlsx"

START_YEAR = 2010
END_YEAR   = 2024

# Три показателя World Bank WDI, необходимые как контрольные переменные
INDICATORS = {
    "NY.GDP.MKTP.CD": "gdp_current_usd",
    "NV.IND.MANF.CD": "manufacturing_va_current_usd",
    "NV.IND.MANF.ZS": "manufacturing_share_gdp",
}

## 1. Загрузка данных World Bank WDI

`download_wdi_indicator` обращается к официальному World Bank API и возвращает длинную таблицу (длинном табличном формате) для одного показателя за 2010–2024.

In [2]:
def download_wdi_indicator(indicator_code, variable_name):
    """Скачивает один показатель World Bank WDI через CSV-API.
    Возвращает длинную (tidy) таблицу: Country Name, Country Code, year, value.
    """
    url = (
        f"https://api.worldbank.org/v2/country/all/indicator/"
        f"{indicator_code}?downloadformat=csv"
    )
    response = requests.get(url)
    response.raise_for_status()

    z = zipfile.ZipFile(io.BytesIO(response.content))
    data_file = [n for n in z.namelist() if n.startswith("API_") and n.endswith(".csv")][0]
    df = pd.read_csv(z.open(data_file), skiprows=4)

    year_cols = [str(y) for y in range(START_YEAR, END_YEAR + 1)]
    df = df[["Country Name", "Country Code"] + year_cols]
    df = df.melt(
        id_vars=["Country Name", "Country Code"],
        value_vars=year_cols,
        var_name="year",
        value_name=variable_name,
    )
    df["year"] = df["year"].astype(int)
    return df

## 2. Слияние с панелью экспорта

Объединяем три показателя WDI и делаем левое объединение с панелью WITS. Ключевой шаг — маппинг названий стран: WITS и World Bank используют разные транслитерации и официальные наименования (например, Vietnam → Viet Nam, Turkey → Turkiye). Несопоставленные страны логируются в отдельный лист Excel.

In [3]:
# Загружаем панель экспорта
panel = pd.read_excel(INPUT_FILE, sheet_name="final_panel_balanced")

# Скачиваем и объединяем все три показателя WDI
controls = None
for indicator_code, variable_name in INDICATORS.items():
    temp = download_wdi_indicator(indicator_code, variable_name)
    controls = temp if controls is None else controls.merge(
        temp, on=["Country Name", "Country Code", "year"], how="outer")

# Маппинг названий стран WITS → World Bank
# (различия в транслитерации и официальных наименованиях)
name_map = {
    "Hong Kong, China":              "Hong Kong SAR, China",
    "Vietnam":                       "Viet Nam",
    "Macao":                         "Macao SAR, China",
    "Serbia, FR(Serbia/Montenegro)": "Serbia",
    "Turkey":                        "Turkiye",
    "Czech Republic":                "Czechia",
    "Venezuela":                     "Venezuela, RB",
    "Other Asia, nes":               None,   # агрегат WITS — не сопоставляется
}

panel["wb_country_name"] = panel["partner"].map(name_map).fillna(panel["partner"])
panel.loc[panel["partner"] == "Other Asia, nes", "wb_country_name"] = None

df = panel.merge(
    controls,
    left_on=["wb_country_name", "year"],
    right_on=["Country Name", "year"],
    how="left",
)

# Логарифмы контролей для использования в регрессиях
df["ln_gdp"]              = np.log(df["gdp_current_usd"])
df["ln_manufacturing_va"] = np.log(df["manufacturing_va_current_usd"])

# Диагностика: страны без GDP из World Bank
unmatched = (
    df[df["gdp_current_usd"].isna()][["partner", "wb_country_name"]]
    .drop_duplicates().sort_values("partner")
)
print("Страны/категории без GDP из World Bank:")
print(unmatched.to_string(index=False))

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="panel_with_controls", index=False)
    unmatched.to_excel(writer, sheet_name="unmatched_countries", index=False)

print(f"\nСохранено: {OUTPUT_FILE}")

Страны/категории без GDP из World Bank:
        partner wb_country_name
Other Asia, nes             NaN



Сохранено: ../data/china_ree_exports_with_controls.xlsx


## 3. Санитарный тест

Быстрая OLS-регрессия с контролями — проверка того, что слияние прошло корректно и все нужные переменные доступны. Это не основная спецификация.

In [4]:
# Быстрый санитарный тест: OLS с контролями на полной панели
# Цель — убедиться, что переменные корректно слились и регрессия сходится
df_val = pd.read_excel(OUTPUT_FILE, sheet_name="panel_with_controls")
df_val["post2016_high_exposure_top15"] = df_val["post2016"] * df_val["high_exposure_top15"]
reg = df_val.dropna(subset=["ln_export_value", "ln_gdp", "manufacturing_share_gdp"])

model = smf.ols(
    formula="ln_export_value ~ post2016_high_exposure_top15 + ln_gdp + manufacturing_share_gdp + C(partner) + C(year)",
    data=reg,
).fit(cov_type="cluster", cov_kwds={"groups": reg["partner"]})

coef = model.params.get("post2016_high_exposure_top15", float("nan"))
pval = model.pvalues.get("post2016_high_exposure_top15", float("nan"))
print(f"Санитарная проверка — post2016_high_exposure_top15: β = {coef:.3f}, p = {pval:.4f}")
print(f"N = {int(model.nobs)}, adj. R² = {model.rsquared_adj:.3f}")

Санитарная проверка — post2016_high_exposure_top15: β = -3.639, p = 0.0000
N = 1145, adj. R² = 0.743


## Результат

### Слияние данных

- Источник панели: WITS / UN Comtrade, HS 284690, 2010–2024
- Контрольные переменные: World Bank WDI (ВВП, добавленная стоимость обрабатывающей промышленности, доля обрабатывающей промышленности)
- Слияние: левое объединение по стране-партнёру и году; страны без совпадения сохранены в лист `unmatched_countries`

### Санитарный тест (OLS с контролями)

Регрессия `ln_export_value ~ Post2016 x HighExposure + ln_gdp + manufacturing_share_gdp + FE`
запускается исключительно для проверки корректности слияния данных — не как основная спецификация.

Основные результаты с контролями и без них представлены в `run_ree_model_selection_v3.ipynb`
(батарея тестов робастности, секция 6).

> Контрольные переменные WDI имеют пропуски в последних годах для ряда стран,
> что сокращает выборку при их включении. Именно поэтому основная спецификация
> оценивается без контролей, а версия с контролями — как проверка устойчивости.